# Teste de Conexão com PostgreSQL

Este notebook testa a conexão com PostgreSQL para identificar e corrigir problemas de conectividade.


In [1]:
# Setup do Spark
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Para o Spark se estiver rodando
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("TestePostgreSQL") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/home/tavares/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

print("✅ Spark configurado!")


✅ Spark configurado!


## Teste 1: Verificar se o driver JDBC está disponível


In [2]:
# Verificar se o driver JDBC está disponível
print("📊 Verificando driver JDBC do PostgreSQL...")

try:
    driver_class = spark.sparkContext._jvm.Class.forName("org.postgresql.Driver")
    print("✅ Driver JDBC do PostgreSQL encontrado!")
    print(f"   Classe: {driver_class.getName()}")
except Exception as e:
    print(f"❌ Erro ao carregar driver: {e}")
    print("💡 O driver pode não estar no classpath do Spark")


📊 Verificando driver JDBC do PostgreSQL...
✅ Driver JDBC do PostgreSQL encontrado!
   Classe: org.postgresql.Driver


## Teste 2: Verificar conectividade de rede


In [3]:
# Testar diferentes hostnames
import socket

hostnames_to_test = [
    "postgres-erp",
    "localhost",
    "172.16.241.3"
]

port = 5432

print("📊 Testando conectividade de rede...")
for hostname in hostnames_to_test:
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(2)
        result = sock.connect_ex((hostname, port))
        sock.close()
        if result == 0:
            print(f"✅ {hostname}:{port} - Porta acessível")
        else:
            print(f"❌ {hostname}:{port} - Porta não acessível")
    except Exception as e:
        print(f"❌ {hostname}:{port} - Erro: {e}")


📊 Testando conectividade de rede...
✅ postgres-erp:5432 - Porta acessível
❌ localhost:5432 - Porta não acessível
✅ 172.16.241.3:5432 - Porta acessível


## Teste 3: Tentar conexão JDBC com diferentes hostnames


In [4]:
# Configuração da conexão
jdbc_port = 5432
jdbc_database = "northwind"
jdbc_username = "postgres"
jdbc_password = "postgres"

connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver"
}

hostnames_to_test = [
    "postgres-erp",
    "localhost",
    "172.16.241.3"
]

print("📊 Testando conexão JDBC...")
df_postgres = None
hostname_funcionou = None

for hostname in hostnames_to_test:
    try:
        jdbc_url = f"jdbc:postgresql://{hostname}:{jdbc_port}/{jdbc_database}"
        print(f"\n🔄 Tentando conectar com: {jdbc_url}")
        
        query = "(SELECT COUNT(*) as total FROM customers) AS test"
        df_test = spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=connection_properties
        )
        
        # Testar a conexão
        result = df_test.collect()[0]['total']
        print(f"✅ Conexão bem-sucedida com {hostname}!")
        print(f"   Total de registros na tabela customers: {result}")
        
        hostname_funcionou = hostname
        
        # Agora ler a tabela completa
        query_full = "(SELECT * FROM customers LIMIT 5) AS customers"
        df_postgres = spark.read.jdbc(
            url=jdbc_url,
            table=query_full,
            properties=connection_properties
        )
        
        print("\n📊 Amostra dos dados:")
        df_postgres.show(truncate=False)
        
        break
        
    except Exception as e:
        print(f"❌ Falhou com {hostname}: {str(e)[:200]}")
        continue

if df_postgres is None:
    print("\n⚠️ Não foi possível conectar com nenhum hostname")
    print("💡 Verifique se o container postgres-erp está rodando")
    print("💡 Execute: docker ps | grep postgres")


📊 Testando conexão JDBC...

🔄 Tentando conectar com: jdbc:postgresql://postgres-erp:5432/northwind
✅ Conexão bem-sucedida com postgres-erp!
   Total de registros na tabela customers: 91

📊 Amostra dos dados:
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|company_name                      |contact_name      |contact_title       |address                      |city       |region|postal_code|country|phone         |fax           |
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|ALFKI      |Alfreds Futterkiste               |Maria Anders      |Sales Representative|Obere Str. 57                |Berlin     |null  |12209      |Germany|030-0074321   |030-0076545   |
|ANATR      |Ana Trujillo Emparedados y 

## Teste 4: Se conectou, fazer JOIN com dados do Iceberg


In [5]:
if df_postgres is not None and hostname_funcionou:
    print(f"✅ Usando hostname: {hostname_funcionou}")
    
    # Criar view temporária
    df_postgres.createOrReplaceTempView("customers_postgres")
    
    # Verificar se há correspondência com vendas
    print("\n📊 Verificando correspondência com dados de vendas...")
    
    # Pegar alguns customer_id do PostgreSQL
    sample_customers = spark.sql("SELECT customer_id FROM customers_postgres LIMIT 5").collect()
    customer_ids = [row['customer_id'] for row in sample_customers]
    
    print(f"📊 Customer IDs do PostgreSQL: {customer_ids}")
    
    # Criar vendas de exemplo com esses customer_id
    print("\n🔄 Criando vendas de exemplo para demonstrar JOIN...")
    
    for i, cust_id in enumerate(customer_ids, start=2000000):
        spark.sql(f"""
            INSERT INTO hadoop_catalog.default.vendas_ecommerce
            VALUES (
                {i},
                'Produto Demo',
                'Eletrônicos',
                1,
                100.00,
                CURRENT_DATE(),
                '{cust_id}',
                101,
                0.0,
                'online'
            )
        """)
    
    print(f"✅ Criadas {len(customer_ids)} vendas de exemplo")
    
    # Fazer JOIN
    print("\n📊 JOIN entre PostgreSQL e Iceberg:")
    spark.sql("""
        SELECT 
            c.customer_id,
            c.company_name,
            c.city,
            c.country,
            COUNT(v.venda_id) as total_vendas,
            ROUND(SUM(v.preco_unitario * v.quantidade), 2) as receita_total
        FROM customers_postgres c
        INNER JOIN hadoop_catalog.default.vendas_ecommerce v
            ON c.customer_id = v.cliente_id
        GROUP BY c.customer_id, c.company_name, c.city, c.country
        ORDER BY receita_total DESC
    """).show(truncate=False)
    
    print("\n✅ Teste de integração concluído com sucesso!")
else:
    print("\n⚠️ Não foi possível testar a integração - conexão não estabelecida")


✅ Usando hostname: postgres-erp

📊 Verificando correspondência com dados de vendas...
📊 Customer IDs do PostgreSQL: ['ALFKI', 'ANATR', 'ANTON', 'AROUT', 'BERGS']

🔄 Criando vendas de exemplo para demonstrar JOIN...
✅ Criadas 5 vendas de exemplo

📊 JOIN entre PostgreSQL e Iceberg:
+-----------+----------------------------------+-----------+-------+------------+-------------+
|customer_id|company_name                      |city       |country|total_vendas|receita_total|
+-----------+----------------------------------+-----------+-------+------------+-------------+
|ANATR      |Ana Trujillo Emparedados y helados|México D.F.|Mexico |1           |100.0        |
|ALFKI      |Alfreds Futterkiste               |Berlin     |Germany|1           |100.0        |
|ANTON      |Antonio Moreno Taquería           |México D.F.|Mexico |1           |100.0        |
|AROUT      |Around the Horn                   |London     |UK     |1           |100.0        |
|BERGS      |Berglunds snabbköp                